In [0]:
# FRED lakehouse pipeline (Lakeflow Spark Declarative Pipelines / DLT)

"""
Bronze : Auto Loader incrementally ingests raw JSON envelopes from the
         landing volume (observations + series metadata) into Delta.
Silver : Parses, types, and deduplicates observations to one row per
         (series_id, observation_date), keeping the latest fetch; builds
         dim_series from FRED's own metadata. Data-quality expectations
         are enforced here.
Gold   : (starter) per-series coverage/freshness summary.
"""

import dlt
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Configure
CATALOG = spark.conf.get("app.catalog")
LANDING = f"/Volumes/{CATALOG}/landing/fred"

In [0]:
# Bronze


@dlt.table(
    name=f"{CATALOG}.bronze.fred_observations_raw",
    comment="Raw FRED observation envelopes ingested from the landing "
            "volume via Auto Loader. One row per landed file.",
    table_properties={"quality": "bronze"},
)
def fred_observations_raw():
    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.partitionColumns", "")  # avoid series_id collision
        .load(f"{LANDING}/observations/")
        .select(
            "*",
            F.col("_metadata.file_path").alias("_source_file"),
            F.current_timestamp().alias("_ingest_ts"),
        )
    )
 
 
@dlt.table(
    name=f"{CATALOG}.bronze.fred_series_metadata_raw",
    comment="Raw FRED series-metadata envelopes (title, units, frequency) "
            "ingested from the landing volume via Auto Loader.",
    table_properties={"quality": "bronze"},
)
def fred_series_metadata_raw():
    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.partitionColumns", "")
        .load(f"{LANDING}/series_metadata/")
        .select(
            "*",
            F.col("_metadata.file_path").alias("_source_file"),
            F.current_timestamp().alias("_ingest_ts"),
        )
    )

In [0]:
# Silver

@dlt.table(
    name=f"{CATALOG}.silver.fred_observations",
    comment="One row per (series_id, observation_date), latest fetch wins. "
            "FRED '.' missing-value markers removed; values typed.",
    table_properties={"quality": "silver"},
)
@dlt.expect_or_fail("series_id_present", "series_id IS NOT NULL")
@dlt.expect_all_or_drop({
    "valid_observation_date": "observation_date IS NOT NULL",
    "no_future_dates":        "observation_date <= current_date()",
    "value_is_numeric":       "value IS NOT NULL",
})
def fred_observations():
    raw = spark.read.table(f"{CATALOG}.bronze.fred_observations_raw")
 
    exploded = raw.select(
        "series_id",
        F.col("group").alias("series_group"),
        F.to_timestamp("fetched_at").alias("fetched_at"),
        F.explode("response.observations").alias("obs"),
    )
 
    typed = (
        exploded
        .where(F.col("obs.value") != ".")          # FRED missing-value marker
        .select(
            "series_id",
            "series_group",
            "fetched_at",
            F.to_date("obs.date").alias("observation_date"),
            # Convert B2310NCBA from millions to billions
            F.when(F.col("series_id") == "B2310NCBA",
                   F.col("obs.value").cast("double") / 1000)
             .otherwise(F.col("obs.value").cast("double"))
             .alias("value"),
            F.to_date("obs.realtime_start").alias("realtime_start"),
            F.to_date("obs.realtime_end").alias("realtime_end"),
        )
    )
 
    # Latest fetch wins per (series_id, observation_date)
    latest_first = Window.partitionBy("series_id", "observation_date") \
                         .orderBy(F.col("fetched_at").desc())
    return (
        typed
        .withColumn("_rn", F.row_number().over(latest_first))
        .where(F.col("_rn") == 1)
        .drop("_rn")
    )
 
 
@dlt.table(
    name=f"{CATALOG}.silver.dim_series",
    comment="Series reference data from FRED's own metadata endpoint. "
            "One row per series, latest metadata fetch wins.",
    table_properties={"quality": "silver"},
)
@dlt.expect_or_fail("series_id_present", "series_id IS NOT NULL")
def dim_series():
    raw = spark.read.table(f"{CATALOG}.bronze.fred_series_metadata_raw")
 
    exploded = raw.select(
        F.to_timestamp("fetched_at").alias("fetched_at"),
        F.explode("series").alias("s"),
    )
 
    latest_first = Window.partitionBy("s.id") \
                         .orderBy(F.col("fetched_at").desc())
    return (
        exploded
        .withColumn("_rn", F.row_number().over(latest_first))
        .where(F.col("_rn") == 1)
        .select(
            F.col("s.id").alias("series_id"),
            F.col("s.title").alias("title"),
            F.col("s.units").alias("units"),
            F.col("s.units_short").alias("units_short"),
            F.col("s.frequency").alias("frequency"),
            F.col("s.frequency_short").alias("frequency_short"),
            F.col("s.seasonal_adjustment_short").alias("seasonal_adjustment"),
            F.col("s.last_updated").alias("source_last_updated"),
            F.col("fetched_at").alias("metadata_fetched_at"),
        )
    )

In [0]:
# Gold

@dlt.table(
    name=f"{CATALOG}.gold.series_coverage",
    comment="Per-series row counts, date coverage, and freshness. Feeds "
            "the data-quality/freshness tile on the dashboard.",
    table_properties={"quality": "gold"},
)
def series_coverage():
    obs = spark.read.table(f"{CATALOG}.silver.fred_observations")
    dim = spark.read.table(f"{CATALOG}.silver.dim_series")
 
    summary = obs.groupBy("series_id", "series_group").agg(
        F.count("*").alias("n_observations"),
        F.min("observation_date").alias("first_observation"),
        F.max("observation_date").alias("latest_observation"),
        F.max("fetched_at").alias("last_fetched_at"),
    )
    return (
        summary.join(dim.select("series_id", "title", "units_short",
                                "frequency_short"),
                     on="series_id", how="left")
    )
